# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset DOI: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset License: {metadata.license}")
print(f"Published on: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets with their @id and name
print("Record Sets in this Croissant dataset:")
record_sets = metadata.recordSet
record_set_ids = []
for rs in record_sets:
    print(f"  @id: {rs['@id']}, name: {rs['name']}")
    record_set_ids.append(rs["@id"])

# For each record set, list its available fields (@id and name)
for rs in record_sets:
    print(f"\nFields for RecordSet '{rs['name']}' (@id: {rs['@id']}):")
    for field in rs['field']:
        fname = field["name"] if "name" in field else field["@id"]
        print(f"  @id: {field['@id']} | name: {fname} | dataType: {field.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We will focus on the primary quantitative table (often the largest record set with protein measurements) for data analysis. You can adapt the selected `record_set_id` and `field_ids` as needed.

In [ ]:
# For demonstration, pick the first (main) record set
main_record_set_id = record_set_ids[0]
print(f"Selected main RecordSet: {main_record_set_id}")

# Load all records for this record set
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Columns in main DataFrame:")
print(df.columns.tolist())

# Show the first 5 rows
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll:
- Select a numeric field using its `@id`.
- Filter records where that field exceeds a threshold.
- Normalize the numeric field.
- Optionally group data by a categorical field (`@id` for group field).

*Update the variables below if you want to explore different fields!*

In [ ]:
# Example: Select a numeric field and categorical field by their @id
# For your specific dataset, replace these with the actual field @ids you want (see data overview above)

# You may need to update these if running on a different version or record set
numeric_field_id = [col for col in df.columns if "coverage" in col.lower() or "abundance" in col.lower() or "count" in col.lower() or "mw" in col.lower() or "weight" in col.lower()]
if numeric_field_id:
    numeric_field = numeric_field_id[0]
else:
    # Fallback to the first field if no obvious numeric field found
    numeric_field = df.columns[0]
print(f"Selected numeric field for analysis: {numeric_field}")

# Filter records; pick an arbitrary threshold for demonstration
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (mean value):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Optionally group by a field likely to be categorical (e.g., 'description' or first string field)
    possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print("Grouped statistics:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field for grouping found.")
else:
    print(f"Field '{numeric_field}' is not numeric; please inspect the DataFrame columns and select an appropriate field.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot a distribution of the numeric field and a bar plot grouped by a categorical variable if detected.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If grouped_df exists, show top groupings (e.g., mean by group)
if 'grouped_df' in locals():
    plt.figure(figsize=(10,5))
    top_n = 10
    ordering = grouped_df.sort_values(numeric_field, ascending=False).head(top_n)
    sns.barplot(data=ordering, x=group_field, y=numeric_field)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"Top {top_n} {group_field} by Mean {numeric_field}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library via its Croissant schema.

- We enumerated the available `@id`s for record sets and their fields, allowing precise referencing.
- We loaded records for a selected record set, filtered and normalized a numeric field, and optionally grouped the data by a categorical field.
- Visualizations were produced to examine distributions and relationships in the data.

This approach makes it easy to extend the workflow to any other Croissant dataset. For further analysis, explore other record sets, fields, or try advanced visualizations and machine learning models.
